In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score
from preprocessing import get_features_and_target
from visualizer import plot_visualizer
import plotly.graph_objects as go
from tabpfn import TabPFNRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
import xgboost as xgb
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor

In [2]:
import huggingface_hub
huggingface_hub.login()

# Getting Dataframe

In [38]:
# Load the training and development datasets
df = pd.read_csv("data/Data_RSW.csv")
df = df[df["Category"] == "Bad"]
df = df.drop_duplicates(subset="Sample ID", keep="last")

In [15]:
def compute_interfacial_failure(df):
     t = df[['Thickness A (mm)', 'Thickness B (mm)']].min(axis=1) 
     f_pull = 1 * (np.pi/4) * (4 * np.sqrt(t))**2 * (0.7 * 365) 
     return np.round(f_pull, 1) 

def compute_pullout_failure(df):
     t = df[['Thickness A (mm)', 'Thickness B (mm)']].min(axis=1) 
     f_pull = np.pi * ((4 * np.sqrt(t)) + 2*t)*t*365 
     return np.round(f_pull, 1) 

In [39]:
df['Interfacial_Failure'] = compute_interfacial_failure(df) 
df['Pullout_Failure'] = compute_pullout_failure(df) 



In [35]:
df.head(5)

,Sample ID,Pressure (PSI),Welding Time (ms),Angle (Deg),Force (N),Current (A),Thickness A (mm),Thickness B (mm),Material,PullTest (N),NuggetDiameter (mm),Category,Comments,Interfacial_Failure,Pullout_Failure
18,2,35,1500,0,52.25,2014.73,0.920,0.925,AISI 1010 carbon steel,5346.4,3.34,Good,DOE,2953.9,5988.6
33,3,95,1500,0,123.36,1063.83,0.944,0.927,AISI 1010 carbon steel,4293.8,4.43,Good,DOE,2976.3,6064.5
55,6,95,1500,0,124.19,1045.90,0.918,0.925,AISI 1010 carbon steel,4161.4,4.35,Good,DOE,2947.4,5967.0
70,7,35,1500,0,63.82,1137.29,0.930,0.937,AISI 1010 carbon steel,3897.5,4.72,Good,DOE,2986.0,6097.2
92,10,35,1500,0,50.62,901.01,0.933,0.929,AISI 1010 carbon steel,4174.4,3.77,Good,DOE,2982.7,6086.3


In [40]:
df["errorpl"] = df["PullTest (N)"] - df["Pullout_Failure"]
df["errorif"] = df["PullTest (N)"] - df["Interfacial_Failure"]
maepl_list = []
rmsepl_list = []

maeif_list = []
rmseif_list = []

for i in df["Sample ID"]:
    maepl = mean_absolute_error(df["PullTest (N)"], df["Pullout_Failure"]) 
    rmsepl = root_mean_squared_error(df["PullTest (N)"], df["Pullout_Failure"]) 

    rmsepl_list.append(rmsepl) 
    maepl_list.append(maepl)

    maeif = mean_absolute_error(df["PullTest (N)"], df["Interfacial_Failure"]) 
    rmseif = root_mean_squared_error(df["PullTest (N)"], df["Interfacial_Failure"]) 

    rmseif_list.append(rmseif) 
    maeif_list.append(maeif)

maepl_mean = np.mean(maepl_list) 
rmsepl_mean = np.mean(rmsepl_list) 
maepl_total = np.sum(maepl_list) 
rmsepl_total = np.sum(rmsepl_list) 

maeif_mean = np.mean(maeif_list) 
rmseif_mean = np.mean(rmseif_list) 
maeif_total = np.sum(maeif_list) 
rmseif_total = np.sum(rmseif_list)

# Good Class

In [37]:
errorpl = df["errorpl"].sum()
errorif = df["errorif"].sum()
print(errorpl, errorif)
print(f"MAE Pullout Failure: {maepl_mean}, RMSE Pullout Failure: {rmsepl_mean}")
print(f"MAE Interfacial Failure: {maeif_mean}, RMSE Interfacial Failure: {rmseif_mean}")
print(f"Total MAE Pullout Failure: {maepl_total}, Total RMSE Pullout Failure: {rmsepl_total}")
print(f"Total MAE Interfacial Failure: {maeif_total}, Total RMSE Interfacial Failure: {rmseif_total}")

-119329.20000000003 416945.19999999995
MAE Pullout Failure: 287.93634311512415, RMSE Pullout Failure: 401.83645022742394
MAE Interfacial Failure: 941.1855530474039, RMSE Interfacial Failure: 977.931486449647
Total MAE Pullout Failure: 127555.8, Total RMSE Pullout Failure: 178013.5474507488
Total MAE Interfacial Failure: 416945.19999999995, Total RMSE Interfacial Failure: 433223.6484971936


# Bad Class

In [41]:
errorpl = df["errorpl"].sum()
errorif = df["errorif"].sum()
print(errorpl, errorif)
print(f"MAE Pullout Failure: {maepl_mean}, RMSE Pullout Failure: {rmsepl_mean}")
print(f"MAE Interfacial Failure: {maeif_mean}, RMSE Interfacial Failure: {rmseif_mean}")
print(f"Total MAE Pullout Failure: {maepl_total}, Total RMSE Pullout Failure: {rmsepl_total}")
print(f"Total MAE Interfacial Failure: {maeif_total}, Total RMSE Interfacial Failure: {rmseif_total}")

-67417.40000000001 -10318.399999999998
MAE Pullout Failure: 3210.3523809523813, RMSE Pullout Failure: 3394.974146475654
MAE Interfacial Failure: 605.8857142857142, RMSE Interfacial Failure: 672.2788055419808
Total MAE Pullout Failure: 67417.40000000001, Total RMSE Pullout Failure: 71294.45707598873
Total MAE Interfacial Failure: 12723.599999999999, Total RMSE Interfacial Failure: 14117.854916381595


# Explode Class

In [30]:
errorpl = df["errorpl"].sum()
errorif = df["errorif"].sum()
print(errorpl, errorif)
print(f"MAE Pullout Failure: {maepl_mean}, RMSE Pullout Failure: {rmsepl_mean}")
print(f"MAE Interfacial Failure: {maeif_mean}, RMSE Interfacial Failure: {rmseif_mean}")
print(f"Total MAE Pullout Failure: {maepl_total}, Total RMSE Pullout Failure: {rmsepl_total}")
print(f"Total MAE Interfacial Failure: {maeif_total}, Total RMSE Interfacial Failure: {rmseif_total}")

-17523.899999999998 27750.3
MAE Pullout Failure: 575.009677419355, RMSE Pullout Failure: 833.9135889712843
MAE Interfacial Failure: 948.7774193548389, RMSE Interfacial Failure: 1081.968704305617
Total MAE Pullout Failure: 17825.300000000003, Total RMSE Pullout Failure: 25851.321258109812
Total MAE Interfacial Failure: 29412.100000000006, Total RMSE Interfacial Failure: 33541.029833474124
